In [ ]:
import pandas as pd
import gdown
import zipfile
import os

# Install gdown if not already installed
try:
    import gdown
except ImportError:
    !pip install gdown -qq
    import gdown

# Google Drive file ID from the provided link
file_id = '1yW_7laG0sjsobJIa6PP2cM_neprltYMg'
output_filename = 'downloaded_file.zip'
output_dir = 'dataset'

# Download the file using gdown
print(f"Downloading file with ID {file_id}...")
gdown.download(id=file_id, output=output_filename, quiet=False)
print(f"File downloaded as {output_filename}")

# Unzip the file
try:
    with zipfile.ZipFile(output_filename, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    print(f"File unzipped to directory: {output_dir}")
except Exception as e:
    print(f"Error unzipping the file: {e}")

# You can now list the contents of the unzipped directory to verify
print(f"Contents of '{output_dir}':")
!ls {output_dir}

Downloading...
From (original): https://drive.google.com/uc?id=1yW_7laG0sjsobJIa6PP2cM_neprltYMg
From (redirected): https://drive.google.com/uc?id=1yW_7laG0sjsobJIa6PP2cM_neprltYMg&confirm=t&uuid=0f028859-fff1-46c7-ae03-d34338cfa6f3
To: /content/downloaded_file.zip
100%|██████████| 171M/171M [00:03<00:00, 50.6MB/s]


File downloaded as downloaded_file.zip
File unzipped to directory: dataset
Contents of 'dataset':
data


In [ ]:
import os
import cv2 # For image processing
import numpy as np
from tqdm import tqdm # For progress bar

# Install opencv-python if not already installed
try:
    import cv2
except ImportError:
    !pip install opencv-python-headless -qq
    import cv2

# Install tqdm for progress bar
try:
    from tqdm import tqdm
except ImportError:
    !pip install tqdm -qq
    from tqdm import tqdm

# Define the base directory where images are located
image_base_dir = 'dataset/data'

images = []
labels = []

image_size = (100, 100)

# Iterate through subdirectories (which are the labels)
for label_name in os.listdir(image_base_dir):
    label_dir = os.path.join(image_base_dir, label_name)
    if os.path.isdir(label_dir):
        print(f"Processing images for label: {label_name}")
        # Iterate through image files in each label directory
        for image_filename in tqdm(os.listdir(label_dir)):
            if image_filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_path = os.path.join(label_dir, image_filename)

                # Read the image
                img = cv2.imread(image_path)

                # Check if image was loaded successfully
                if img is None:
                    print(f"Warning: Could not load image {image_path}. Skipping.")
                    continue

                # Convert to grayscale
                # Ignore warnings, as requested, by not trying to catch specific errors here
                gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                # Resize to 100x100 pixels
                resized_img = cv2.resize(gray_img, image_size)

                # Flatten the image
                flattened_img = resized_img.flatten()

                # Normalize pixel values by dividing by 255.0
                normalized_img = flattened_img / 255.0

                images.append(normalized_img)
                labels.append(label_name)

# Convert lists to numpy arrays
images_np = np.array(images)
labels_np = np.array(labels)

print(f"\nShape of images array: {images_np.shape}")
print(f"Shape of labels array: {labels_np.shape}")

# Count images for the label 'without_mask'
without_mask_count = np.sum(labels_np == 'without_mask')
print(f"Number of images with label 'without_mask': {without_mask_count}")


Processing images for label: without_mask


100%|██████████| 3828/3828 [00:02<00:00, 1615.41it/s]


Processing images for label: with_mask


100%|██████████| 3725/3725 [00:05<00:00, 654.59it/s]



Shape of images array: (7553, 10000)
Shape of labels array: (7553,)
Number of images with label 'without_mask': 3828


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

# 1. Encode the labels
# Create a custom mapping for LabelEncoder to ensure 'without_mask' is 1 and 'with_mask' is 0
# LabelEncoder assigns labels alphabetically, so we need to transform it
le = LabelEncoder()
encoded_labels = le.fit_transform(labels_np)

# Check what LabelEncoder assigned
# By default, 'with_mask' (alphabetically first) might be 0 and 'without_mask' might be 1
# We need 'without_mask' = 1 and 'with_mask' = 0
# If le.classes_ is ['with_mask', 'without_mask'], then 'with_mask' is 0, 'without_mask' is 1, which is correct.
# If le.classes_ is ['without_mask', 'with_mask'], then 'without_mask' is 0, 'with_mask' is 1, which is incorrect.

# Custom mapping to ensure 'without_mask' is 1 and 'with_mask' is 0
# If 'without_mask' is mapped to 0 by default, we invert the labels
if le.transform(['without_mask'])[0] == 0:
    encoded_labels = 1 - encoded_labels # Invert 0 to 1 and 1 to 0

print(f"Original labels: {np.unique(labels_np)}")
print(f"Encoded labels (0: with_mask, 1: without_mask): {np.unique(encoded_labels)}")


# 2. Split the dataset
X_train, X_test, y_train, y_test = train_test_split(images_np, encoded_labels, test_size=0.2, random_state=0)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

# 3. Train a LogisticRegression model
print("\nTraining Logistic Regression model...")
model = LogisticRegression(random_state=0, max_iter=500, tol=0.001, C=10, solver='liblinear') # 'liblinear' is good for small datasets and L1/L2 regularization
model.fit(X_train, y_train)
print("Model training complete.")

# Predict on the test dataset
y_pred = model.predict(X_test)

# 4. Calculate false positive data points
# 'without_mask' is the positive class (1)
# False Positive: Actual is negative (0, 'with_mask'), Predicted is positive (1, 'without_mask')
cm = confusion_matrix(y_test, y_pred)

# The confusion matrix is typically structured as:
# [[True Negative, False Positive],
#  [False Negative, True Positive]]

false_positives = cm[0, 1]

print(f"\nConfusion Matrix:\n{cm}")
print(f"Number of false positive data points on test dataset: {false_positives}")

Original labels: ['with_mask' 'without_mask']
Encoded labels (0: with_mask, 1: without_mask): [0 1]

Shape of X_train: (6042, 10000)
Shape of X_test: (1511, 10000)
Shape of y_train: (6042,)
Shape of y_test: (1511,)

Training Logistic Regression model...
Model training complete.

Confusion Matrix:
[[458 287]
 [220 546]]
Number of false positive data points on test dataset: 287


In [ ]:
import numpy as np
import cv2

def augment_image(images, labels, angles, augmentation_factor):
    np.random.seed(0)

    augmented_images_list = images.tolist()  # Start with original images
    augmented_labels_list = labels.tolist()  # Start with original labels

    image_height, image_width = 100, 100 # Assuming images are 100x100 and flattened

    # Iterate through each original image to create augmented versions
    for i, original_image in enumerate(images):
        for j in range(augmentation_factor):
            # Reshape the flattened image back to 2D for rotation
            img_2d = original_image.reshape(image_height, image_width)

            # Get the rotation angle for this specific augmented image
            # The angles array is flattened, so we need to pick the correct one
            # The index is (i * augmentation_factor) + j
            rotation_angle = angles[(i * augmentation_factor) + j]

            # Get rotation matrix
            center = (image_width // 2, image_height // 2)
            M = cv2.getRotationMatrix2D(center, rotation_angle, 1.0)

            # Perform the rotation
            # Ensure the output size is the same as the input size (100x100)
            rotated_img = cv2.warpAffine(img_2d, M, (image_width, image_height), flags=cv2.INTER_LINEAR)

            # Flatten the rotated image and normalize if necessary (already normalized 0-1)
            flattened_rotated_img = rotated_img.flatten()

            augmented_images_list.append(flattened_rotated_img)
            augmented_labels_list.append(labels[i])

    # Convert lists to NumPy arrays
    augmented_images = np.array(augmented_images_list)
    augmented_labels = np.array(augmented_labels_list)

    return augmented_images, augmented_labels

# Constraints and Tasks:

# 1. Generate angle_of_rotation
augmentation_factor = 2
np.random.seed(0) # Ensure reproducibility for angle generation
num_total_augmented_images = augmentation_factor * len(X_train)
angle_of_rotation = np.random.uniform(low=-180, high=180, size=num_total_augmented_images)

print(f"Shape of angle_of_rotation: {angle_of_rotation.shape}")

# 2. Call the augment_image function
augmented_X_train, augmented_y_train = augment_image(X_train, y_train, angle_of_rotation, augmentation_factor)

print(f"Shape of augmented_X_train: {augmented_X_train.shape}")
print(f"Shape of augmented_y_train: {augmented_y_train.shape}")

# 3. Compute the sum of elements in augmented_labels from index 7000 to 8000 (exclusive of 8000)
sum_of_labels_slice = np.sum(augmented_y_train[7000:8000])
print(f"Sum of augmented_y_train from index 7000 to 8000: {sum_of_labels_slice}")

Shape of angle_of_rotation: (12084,)
Shape of augmented_X_train: (18126, 10000)
Shape of augmented_y_train: (18126,)
Sum of augmented_y_train from index 7000 to 8000: 510


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
import numpy as np

# 1. Train a RandomForestClassifier for feature importance
print("Training RandomForestClassifier for feature selection...")
# Use a reasonable number of estimators for feature importance. Larger n_estimators generally lead to more stable importances.
rf_model = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1) # n_jobs=-1 uses all available cores
rf_model.fit(augmented_X_train, augmented_y_train)
print("RandomForestClassifier training complete.")

# 2. Select top 100 features based on impurity-based feature importances
# SelectFromModel uses feature_importances_ by default
# We set prefit=True because the model is already trained
print("Selecting top 100 features...")
selector = SelectFromModel(rf_model, threshold=-np.inf, max_features=100, prefit=True)

# Transform the augmented training data and test data to include only selected features
augmented_X_train_selected = selector.transform(augmented_X_train)
X_test_selected = selector.transform(X_test)

print(f"Shape of augmented_X_train after feature selection: {augmented_X_train_selected.shape}")
print(f"Shape of X_test after feature selection: {X_test_selected.shape}")

# 3. Fit a Logistic Regression model using the selected top 100 features
print("Training Logistic Regression model with selected features...")
# Re-initialize the Logistic Regression model for clarity, using similar parameters as before
model_selected_features = LogisticRegression(random_state=0, max_iter=500, tol=0.001, C=10, solver='liblinear')
model_selected_features.fit(augmented_X_train_selected, augmented_y_train)
print("Logistic Regression model training with selected features complete.")

# 4. Predict on the feature-selected test dataset
y_pred_selected = model_selected_features.predict(X_test_selected)

# 5. Calculate the number of misclassified data points from the test data
# Misclassified points are where y_test != y_pred_selected
misclassified_data_points = np.sum(y_test != y_pred_selected)

print(f"\nNumber of misclassified data points on the test dataset with top 100 features: {misclassified_data_points}")

# Optionally, also print confusion matrix for full context
cm_selected_features = confusion_matrix(y_test, y_pred_selected)
print(f"Confusion Matrix with top 100 features:\n{cm_selected_features}")

Training RandomForestClassifier for feature selection...
RandomForestClassifier training complete.
Selecting top 100 features...
Shape of augmented_X_train after feature selection: (18126, 100)
Shape of X_test after feature selection: (1511, 100)
Training Logistic Regression model with selected features...
Logistic Regression model training with selected features complete.

Number of misclassified data points on the test dataset with top 100 features: 644
Confusion Matrix with top 100 features:
[[339 406]
 [238 528]]


In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Perform PCA with n_components=100 on the augmented training data
print("Performing PCA on augmented training data...")
pca = PCA(n_components=100, random_state=0)
augmented_X_train_pca = pca.fit_transform(augmented_X_train)
X_test_pca = pca.transform(X_test)

print(f"Shape of augmented_X_train after PCA: {augmented_X_train_pca.shape}")
print(f"Shape of X_test after PCA: {X_test_pca.shape}")

# 2. Train a RandomForestClassifier with random_state=0
print("Training RandomForestClassifier on PCA-transformed data...")
rf_pca_model = RandomForestClassifier(random_state=0, n_jobs=-1) # Use all available cores
rf_pca_model.fit(augmented_X_train_pca, augmented_y_train)
print("RandomForestClassifier training complete.")

# 3. Predict on the PCA-transformed test dataset
y_pred_pca_rf = rf_pca_model.predict(X_test_pca)

# 4. Calculate the accuracy score on the test dataset
accuracy_pca_rf = accuracy_score(y_test, y_pred_pca_rf)

print(f"\nAccuracy score on the test dataset after PCA + RandomForest: {accuracy_pca_rf:.4f}")

Performing PCA on augmented training data...
Shape of augmented_X_train after PCA: (18126, 100)
Shape of X_test after PCA: (1511, 100)
Training RandomForestClassifier on PCA-transformed data...
RandomForestClassifier training complete.

Accuracy score on the test dataset after PCA + RandomForest: 0.7723
